In [12]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier

# Paths
ROOT = os.getcwd()
DATA_PROCESSED = os.path.join(ROOT, "data", "processed")

# Load dataset
data_path = os.path.join(DATA_PROCESSED, "pit_within_2_laps.parquet")
df = pd.read_parquet(data_path)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (2586, 15)


,LapNumber,LapTimeSec,S1Sec,S2Sec,S3Sec,TyreLife,LapDeltaPrev,LapMean3,LapStd3,PitWithinNextNLaps,Driver,GP,Season,Compound_MEDIUM,Compound_SOFT
0,2.0,100.430,31.765,43.909,24.756,2.0,-0.287,100.430000,0.202940,0,ALB,Bahrain,2023,False,True
1,3.0,100.143,31.660,43.782,24.701,3.0,-0.287,100.286500,0.202940,0,ALB,Bahrain,2023,False,True
2,4.0,99.761,31.226,43.808,24.727,4.0,-0.382,100.111333,0.335622,0,ALB,Bahrain,2023,False,True
3,13.0,98.649,31.512,42.949,24.188,2.0,-1.112,99.517667,0.776155,0,ALB,Bahrain,2023,False,True
4,14.0,98.472,31.104,43.144,24.224,3.0,-0.177,98.960667,0.698736,0,ALB,Bahrain,2023,False,True


In [13]:
# Target
y = df["PitWithinNextNLaps"].astype(int)

# Features (drop identifiers)
X = df.drop(columns=["PitWithinNextNLaps", "Driver", "Season"])
groups = df["GP"]  # group by race

print("Features shape:", X.shape)


Features shape: (2586, 12)


In [14]:
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (1760, 12)
Test size: (826, 12)


In [15]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


ValueError: could not convert string to float: 'Great Britain'

In [5]:
from sklearn.metrics import accuracy_score, f1_score

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Baseline Accuracy:", round(accuracy, 3))
print("Baseline F1-score:", round(f1, 3))


AttributeError: 'RandomForestClassifier' object has no attribute 'estimators_'

In [6]:
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

y_pred = model.predict(X_test)

print("y_pred sample:", y_pred[:10])
print("y_test sample:", y_test.iloc[:10].to_list())

from sklearn.metrics import accuracy_score, f1_score
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Baseline Accuracy:", accuracy)
print("Baseline F1-score:", f1)


X_test shape: (826, 12)
y_test shape: (826,)


AttributeError: 'RandomForestClassifier' object has no attribute 'estimators_'

In [16]:
# Target
y = df["PitWithinNextNLaps"].astype(int)

# Groups (we use GP ONLY for splitting, not as a feature)
groups = df["GP"]

# Features: drop GP as well (IMPORTANT)
X = df.drop(columns=["PitWithinNextNLaps", "Driver", "Season", "GP"])

print("X dtypes:")
print(X.dtypes)
print("X shape:", X.shape)


X dtypes:
LapNumber          float64
LapTimeSec         float64
S1Sec              float64
S2Sec              float64
S3Sec              float64
TyreLife           float64
LapDeltaPrev       float64
LapMean3           float64
LapStd3            float64
Compound_MEDIUM       bool
Compound_SOFT         bool
dtype: object
X shape: (2586, 11)


In [17]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# Split by GP (to avoid leakage)
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Train model
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Predict + evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Baseline Accuracy:", round(accuracy, 3))
print("Baseline F1-score:", round(f1, 3))
print("Test label balance (0/1):")
print(y_test.value_counts())


Baseline Accuracy: 1.0
Baseline F1-score: 0.0
Test label balance (0/1):
PitWithinNextNLaps
0    826
Name: count, dtype: int64


c:\Users\khadi\OneDrive\Desktop\ResearchProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [18]:
from sklearn.model_selection import train_test_split

# features/target as before
y = df["PitWithinNextNLaps"].astype(int)
X = df.drop(columns=["PitWithinNextNLaps", "Driver", "Season", "GP"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Test label balance:\n", y_test.value_counts())


Test label balance:
 PitWithinNextNLaps
0    515
1      3
Name: count, dtype: int64


In [19]:
print(df["PitWithinNextNLaps"].value_counts())
print("Positive rate:", df["PitWithinNextNLaps"].mean())



PitWithinNextNLaps
0    2572
1      14
Name: count, dtype: int64
Positive rate: 0.005413766434648105


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

y = df["PitWithinNextNLaps"].astype(int)
X = df.drop(columns=["PitWithinNextNLaps", "Driver", "Season", "GP"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Baseline Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Baseline F1:", round(f1_score(y_test, y_pred), 3))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))


Baseline Accuracy: 0.994
Baseline F1: 0.0

Classification Report:

              precision    recall  f1-score   support

           0       0.99      1.00      1.00       515
           1       0.00      0.00      0.00         3

    accuracy                           0.99       518
   macro avg       0.50      0.50      0.50       518
weighted avg       0.99      0.99      0.99       518

